# ▶ Run This Before The Session Starts

> Run all cells in this section **before the audience arrives**. This downloads and loads the models (~3–5 minutes). Once both ✅ messages appear, also pre-run the UMAP cell in Part 3, then collapse this section.


In [ ]:
# Install all required packages
# Uses sys.executable to ensure the correct pip for this kernel/venv is called.
# Works on Google Colab, local Jupyter, and uv-managed venvs.
import sys
!{sys.executable} -m pip install -q \
    'transformers>=4.38.0' \
    'torch>=2.0.0' \
    'gensim>=4.3.0' \
    'umap-learn>=0.5.6' \
    'matplotlib>=3.8.0' \
    'numpy>=1.24.0'
print('✅ All packages installed')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, HTML
from transformers import GPT2Tokenizer, BertTokenizer
import gensim.downloader as api

print('✅ Imports ready')

In [ ]:
print('Loading GPT-2 tokenizer (BPE)...')
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
print('✅ GPT-2 tokenizer loaded')

In [ ]:
print('Loading BERT tokenizer (WordPiece)...')
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print('✅ BERT tokenizer loaded')

In [ ]:
print('Loading word2vec-google-news-300 (~1.6 GB, takes 2–3 minutes)...')
w2v = api.load('word2vec-google-news-300')
print(f'✅ word2vec loaded: {len(w2v):,} words')

---
# 📖 PART 1 — Tokenization

> Two common subword strategies: **BPE** (GPT-2) learns piece boundaries from co-occurrence frequency. **WordPiece** (BERT) does the same but marks continuation subwords with `##`, making splits more explicit. Common words stay whole. Rare or complex words get broken into smaller pieces. Before any meaning is calculated, the model already starts from these internal units.


In [ ]:
def render_tokens(text, tokenizer):
    """Render BPE tokens as styled HTML chip cards with token IDs."""
    tokens = tokenizer.tokenize(text)
    ids = tokenizer.convert_tokens_to_ids(tokens)
    # Ġ = space encoded into token (byte-level BPE). Replace for cleaner display.
    display_tokens = [tok.replace("Ġ", "▪") for tok in tokens]

    chips_html = "".join([
        f"""
        <div style="display:inline-block; margin:6px; text-align:center;">
          <div style="
            background:#0f172a;
            border:2px solid #22d3ee;
            border-radius:20px;
            padding:8px 16px;
            color:#f1f5f9;
            font-family:monospace;
            font-size:15px;
            font-weight:600;
          ">{dtok}</div>
          <div style="
            color:#64748b;
            font-family:monospace;
            font-size:11px;
            margin-top:4px;
          ">ID: {id_}</div>
        </div>"""
        for dtok, id_ in zip(display_tokens, ids)
    ])

    html = f"""
    <div style="
      background:#1e293b;
      border-radius:12px;
      padding:20px 24px;
      margin:12px 0;
      font-family:monospace;
    ">
      <div style="color:#94a3b8; font-size:12px; margin-bottom:12px;">INPUT TEXT</div>
      <div style="color:#e2e8f0; font-size:18px; font-weight:700; margin-bottom:16px;">&quot;{text}&quot;</div>
      <div style="color:#94a3b8; font-size:12px; margin-bottom:8px;">TOKENS ({len(tokens)} piece{"s" if len(tokens) != 1 else ""})</div>
      <div>{chips_html}</div>
    </div>
    """
    return HTML(html)

print('✅ Helper function ready')

In [ ]:
# Story examples — the phrases we revisit throughout the demo
display(render_tokens('river bank', tokenizer))
display(render_tokens('bank account', tokenizer))

In [ ]:
# BPE in action — common words stay whole; rare/complex words get split
display(render_tokens('Nguyễn', tokenizer))
display(render_tokens('tokenization', tokenizer))
display(render_tokens('banking', tokenizer))

In [ ]:
# How many pieces does each input become?
examples = [
    ('river bank',   'story example'),
    ('bank account', 'story example'),
    ('bank',         'single common word'),
    ('banking',      'common word + suffix'),
    ('tokenization', 'longer / abstract word'),
    ('Nguyễn',      'rare / foreign name'),
]

rows_html = ''
for text, label in examples:
    tokens = tokenizer.tokenize(text)
    display_tok = [t.replace('Ġ', '▪') for t in tokens]
    n = len(tokens)
    color = '#22d3ee' if n == 1 else ('#f59e0b' if n == 2 else '#e879f9')
    chips = ' '.join([
        f'<code style="background:#0f172a; border:1px solid #334155; border-radius:6px; '
        f'padding:2px 8px; color:#e2e8f0; font-size:13px;">{t}</code>'
        for t in display_tok
    ])
    rows_html += (
        f'<tr>'
        f'<td style="padding:8px 14px; font-family:monospace; color:#e2e8f0; font-size:14px;">&quot;{text}&quot;</td>'
        f'<td style="padding:8px 14px; color:#64748b; font-size:12px;">{label}</td>'
        f'<td style="padding:8px 14px;">{chips}</td>'
        f'<td style="padding:8px 14px; font-family:monospace; font-weight:700; color:{color}; font-size:18px;">{n}</td>'
        f'</tr>'
    )

html = (
    "<div style='background:#1e293b; border-radius:12px; padding:20px 24px; margin:12px 0;'>"
    "<div style='color:#94a3b8; font-size:12px; margin-bottom:12px;'>"
    "BPE TOKEN COUNT — how many pieces does each input become?</div>"
    "<table style='border-collapse:collapse; width:100%;'>"
    "<thead><tr style='border-bottom:1px solid #334155;'>"
    "<th style='padding:4px 14px; color:#64748b; text-align:left; font-size:11px;'>INPUT</th>"
    "<th style='padding:4px 14px; color:#64748b; text-align:left; font-size:11px;'>TYPE</th>"
    "<th style='padding:4px 14px; color:#64748b; text-align:left; font-size:11px;'>PIECES</th>"
    "<th style='padding:4px 14px; color:#64748b; text-align:left; font-size:11px;'>COUNT</th>"
    "</tr></thead>"
    f"<tbody>{rows_html}</tbody>"
    "</table>"
    "<div style='margin-top:14px; color:#94a3b8; font-size:12px;'>"
    "▪ = space encoded into token &nbsp;|&nbsp; "
    "<span style='color:#22d3ee;'>&#9632; 1 piece</span> &nbsp; "
    "<span style='color:#f59e0b;'>&#9632; 2 pieces</span> &nbsp; "
    "<span style='color:#e879f9;'>&#9632; 3+ pieces</span></div>"
    "</div>"
)
display(HTML(html))

### BPE vs WordPiece — same words, different strategies

> **WordPiece improvement:** continuation subwords are marked with `##`. This makes it explicit which pieces belong together as one word — unlike BPE where pieces just sit next to each other.


In [ ]:
# BPE (GPT-2) vs WordPiece (BERT) — side-by-side for the same words
words_to_compare = ['bank', 'banking', 'tokenization', 'Nguyễn', 'preprocessing']

def tok_chips(tokens, color_continuation):
    html_parts = []
    for tok in tokens:
        is_cont = tok.startswith('##') or tok.startswith('Ġ')
        bg = '#1e1040' if is_cont and color_continuation else '#0f172a'
        border = '#e879f9' if is_cont and color_continuation else '#22d3ee'
        html_parts.append(
            f'<span style="display:inline-block; margin:3px; padding:4px 10px; '
            f'background:{bg}; border:1.5px solid {border}; border-radius:14px; '
            f'font-family:monospace; font-size:13px; color:#f1f5f9;">{tok}</span>'
        )
    return ' '.join(html_parts)

rows_html = ''
for word in words_to_compare:
    bpe = tokenizer.tokenize(word)
    wp  = bert_tokenizer.tokenize(word)
    bpe_display = [t.replace('Ġ', '▪') for t in bpe]
    rows_html += (
        f'<tr>'
        f'<td style="padding:8px 14px; font-family:monospace; color:#e2e8f0; font-size:14px; border-bottom:1px solid #1e293b;">{word}</td>'
        f'<td style="padding:8px 14px; border-bottom:1px solid #1e293b;">{tok_chips(bpe_display, False)}</td>'
        f'<td style="padding:8px 14px; font-family:monospace; color:#94a3b8; font-size:12px; border-bottom:1px solid #1e293b;">{len(bpe)}</td>'
        f'<td style="padding:8px 14px; border-bottom:1px solid #1e293b;">{tok_chips(wp, True)}</td>'
        f'<td style="padding:8px 14px; font-family:monospace; color:#94a3b8; font-size:12px; border-bottom:1px solid #1e293b;">{len(wp)}</td>'
        f'</tr>'
    )

html = (
    "<div style='background:#1e293b; border-radius:12px; padding:20px 24px; margin:12px 0;'>"
    "<div style='color:#94a3b8; font-size:12px; margin-bottom:14px;'>"
    "BPE (GPT-2) vs WordPiece (BERT) — same words, different splitting strategies</div>"
    "<table style='border-collapse:collapse; width:100%;'>"
    "<thead><tr style='border-bottom:2px solid #334155;'>"
    "<th style='padding:6px 14px; color:#64748b; text-align:left; font-size:11px;'>WORD</th>"
    "<th style='padding:6px 14px; color:#22d3ee; text-align:left; font-size:11px;'>BPE pieces (GPT-2)</th>"
    "<th style='padding:6px 14px; color:#22d3ee; text-align:left; font-size:11px;'>#</th>"
    "<th style='padding:6px 14px; color:#e879f9; text-align:left; font-size:11px;'>WordPiece pieces (BERT)</th>"
    "<th style='padding:6px 14px; color:#e879f9; text-align:left; font-size:11px;'>#</th>"
    "</tr></thead>"
    f"<tbody>{rows_html}</tbody>"
    "</table>"
    "<div style='margin-top:14px; color:#94a3b8; font-size:12px;'>"
    "<span style='color:#22d3ee;'>cyan border</span> = standalone piece &nbsp;|&nbsp; "
    "<span style='color:#e879f9;'>magenta border + ## prefix</span> = WordPiece continuation subword"
    "</div></div>"
)
display(HTML(html))

---
# 📐 PART 2 — Word Similarity

> **Words with similar meaning end up close in vector space.** word2vec learned this purely from co-occurrence patterns across billions of text tokens — no human labeling required.


In [ ]:
# ✏️  Change this word live — invite the audience to suggest one!
query_word = "king"

results = w2v.most_similar(query_word, topn=10)

rows = "".join([
    f"""
    <tr>
      <td style="padding:6px 16px; color:#94a3b8; font-family:monospace;">#{i+1}</td>
      <td style="padding:6px 16px; color:#e2e8f0; font-family:monospace; font-weight:600;">{word}</td>
      <td style="padding:6px 16px; color:#22d3ee; font-family:monospace;">{score:.4f}</td>
    </tr>"""
    for i, (word, score) in enumerate(results)
])

html = f"""
<div style="background:#1e293b; border-radius:12px; padding:20px 24px; max-width:460px;">
  <div style="color:#94a3b8; font-size:12px; margin-bottom:4px;">MOST SIMILAR TO</div>
  <div style="color:#f1f5f9; font-size:24px; font-weight:700; margin-bottom:16px; font-family:monospace;">&quot;{query_word}&quot;</div>
  <table style="border-collapse:collapse; width:100%;">
    <thead>
      <tr style="border-bottom:1px solid #334155;">
        <th style="padding:4px 16px; color:#64748b; text-align:left; font-size:11px;">#</th>
        <th style="padding:4px 16px; color:#64748b; text-align:left; font-size:11px;">WORD</th>
        <th style="padding:4px 16px; color:#64748b; text-align:left; font-size:11px;">SIMILARITY</th>
      </tr>
    </thead>
    <tbody>{rows}</tbody>
  </table>
</div>
"""
display(HTML(html))

### Arithmetic on Meaning

> If meaning is geometry, we can **add and subtract** concepts.
>
> The famous example: &nbsp; **king − man + woman = ?**


In [ ]:
results = w2v.most_similar(positive=["king", "woman"], negative=["man"], topn=5)

rows = "".join([
    f"""
    <tr style="background:{"#1e1040" if word.lower()=="queen" else "transparent"}">
      <td style="padding:8px 16px; font-size:20px;">{"👑" if word.lower()=="queen" else ""}</td>
      <td style="padding:8px 16px; color:{"#e879f9" if word.lower()=='queen' else '#e2e8f0'}; font-family:monospace; font-weight:{"700" if word.lower()=='queen' else '400'}; font-size:16px;">{word}</td>
      <td style="padding:8px 16px; color:{"#e879f9" if word.lower()=='queen' else '#22d3ee'}; font-family:monospace;">{score:.4f}</td>
    </tr>"""
    for word, score in results
])

html = f"""
<div style="background:#1e293b; border-radius:12px; padding:20px 24px; max-width:460px;">
  <div style="color:#f1f5f9; font-size:18px; font-weight:700; margin-bottom:4px; font-family:monospace;">
    king &minus; man + woman = ?
  </div>
  <div style="color:#64748b; font-size:12px; margin-bottom:16px;">Top results — word2vec-google-news-300</div>
  <table style="border-collapse:collapse; width:100%;">
    <tbody>{rows}</tbody>
  </table>
</div>
"""
display(HTML(html))

---
# 🗺️ PART 3 — Embedding Map

> **Let's see the whole neighborhood at once.** UMAP compresses 300-dimensional word vectors into 2D so we can visualize the semantic clusters.

> ⚠️ **Presenter note:** Run this cell during SETUP and **do not re-run it live** — UMAP is non-deterministic and the layout will shift between runs.


In [ ]:
import umap

# Curated word list — 5 semantic categories
categories = {
    'Royalty':     {'words': ['king', 'queen', 'prince', 'princess', 'throne', 'crown'],             'color': '#f59e0b'},
    'Animals':     {'words': ['dog', 'cat', 'wolf', 'horse', 'lion', 'tiger'],                       'color': '#2dd4bf'},
    'Geography':   {'words': ['river', 'ocean', 'mountain', 'forest', 'valley', 'beach'],            'color': '#4ade80'},
    'Professions': {'words': ['doctor', 'nurse', 'engineer', 'teacher', 'lawyer', 'scientist'],      'color': '#a78bfa'},
    'Finance':     {'words': ['money', 'loan', 'credit', 'investment', 'bank', 'currency'],          'color': '#fb923c'},
}

all_words, all_colors, all_labels = [], [], []
for cat_name, cat in categories.items():
    for word in cat['words']:
        if word in w2v:
            all_words.append(word)
            all_colors.append(cat['color'])
            all_labels.append(cat_name)

vectors = np.array([w2v[w] for w in all_words])

reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=8, min_dist=0.3)
embedding = reducer.fit_transform(vectors)

fig, ax = plt.subplots(figsize=(13, 8))
fig.patch.set_facecolor('#0f172a')
ax.set_facecolor('#0f172a')

ax.scatter(embedding[:, 0], embedding[:, 1], c=all_colors, s=130, alpha=0.92, zorder=3)

for i, word in enumerate(all_words):
    is_bank = word == 'bank'
    ax.annotate(
        word,
        (embedding[i, 0], embedding[i, 1]),
        xytext=(7, 7), textcoords='offset points',
        fontsize=11,
        color='#e879f9' if is_bank else '#e2e8f0',
        fontfamily='monospace',
        fontweight='bold' if is_bank else 'normal',
    )

legend_patches = [
    mpatches.Patch(color=cat['color'], label=cat_name)
    for cat_name, cat in categories.items()
]
ax.legend(
    handles=legend_patches, loc='lower right',
    facecolor='#1e293b', edgecolor='#334155',
    labelcolor='#e2e8f0', fontsize=11
)

ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_edgecolor('#334155')

ax.set_title(
    'Word Embedding Space — UMAP 2D Projection',
    color='#94a3b8', fontsize=13, pad=16, fontfamily='monospace'
)

plt.tight_layout()
plt.show()

---
# 🔁 Closing — Back to the Puzzle

> We started with one question:
>
> *Why does the word **bank** seem simple to us, but difficult for a model?*
>
> Let's look at the evidence we collected.


In [ ]:
# Same word — two contexts — same BPE token pieces, but different contextual embeddings
display(render_tokens('river bank', tokenizer))
display(render_tokens('bank account', tokenizer))

---
## Why is *bank* difficult for a model?

| Layer | What we saw |
|---|---|
| **Tokenization** | BPE splits text into deterministic subword pieces — common words stay whole, rare words split |
| **Embeddings** | *bank* sits near *money* and *loan* in the Finance cluster |
| **Context** | Static models assign one fixed vector; contextual models read meaning from the whole sentence |

> **Same surface. Different meaning. Different representation.**
>
> *Modern NLP is representation, geometry, and context at scale.*
